# 01 — Data Ingestion

**Goal:** Load HM Land Registry Price Paid Data (2021–2024), filter to London transactions only, save as Parquet for downstream cleaning.

**Input:** `data/raw/pp-2021.csv`, `pp-2022.csv`, `pp-2023.csv`, `pp-2024.csv`  
**Output:** `data/processed/london_transactions_raw.parquet`

**Filtering logic:** London postcodes only — prefixes E, N, W, SW, SE, EC, WC.

---

## Why this matters
- Raw data is ~600MB across 4 files (England & Wales)
- We filter at ingestion to reduce memory footprint by ~85%
- Parquet output is ~10x smaller and faster to read in later notebooks

## Imports

In [1]:
import pandas as pd
import os
import glob
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print(f"pandas version: {pd.__version__}")
print(f"Working directory: {os.getcwd()}")

pandas version: 3.0.3
Working directory: c:\Users\irfan\london-housing-analytics\notebooks


##  Define Paths and Columns

In [2]:
# Project paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

# Make sure processed dir exists
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Land Registry column names (files have no headers)
COLUMNS = [
    'transaction_id', 'price', 'date', 'postcode',
    'property_type', 'new_build', 'tenure', 'paon',
    'saon', 'street', 'locality', 'town', 'district',
    'county', 'ppd_type', 'record_status'
]

# London postcode prefixes
LONDON_PREFIXES = ('E', 'N', 'W', 'SW', 'SE', 'EC', 'WC')

print(f"Raw data folder: {RAW_DIR}")
print(f"Processed data folder: {PROCESSED_DIR}")
print(f"Files in raw folder:")
for f in RAW_DIR.glob('*.csv'):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  - {f.name} ({size_mb:.1f} MB)")

Raw data folder: c:\Users\irfan\london-housing-analytics\data\raw
Processed data folder: c:\Users\irfan\london-housing-analytics\data\processed
Files in raw folder:
  - pp-2021.csv (212.8 MB)
  - pp-2022.csv (179.0 MB)
  - pp-2023.csv (143.1 MB)
  - pp-2024.csv (154.0 MB)


## London Filter Function

In [3]:
def filter_london(df: pd.DataFrame) -> pd.DataFrame:
    """
    Filter dataframe to London postcodes only.
    London postcode districts start with: E, EC, N, NW, SE, SW, W, WC
    
    Note: We use a stricter prefix check to avoid false matches.
    """
    # Drop rows with null postcodes first
    df = df.dropna(subset=['postcode']).copy()
    
    # Get postcode area (letters before first digit)
    # e.g. "SW1A 1AA" -> "SW", "E14 5AB" -> "E"
    df['postcode_area'] = df['postcode'].str.extract(r'^([A-Z]+)', expand=False)
    
    # Filter to London areas only
    london_areas = ['E', 'EC', 'N', 'NW', 'SE', 'SW', 'W', 'WC']
    df_london = df[df['postcode_area'].isin(london_areas)].copy()
    
    # Drop helper column
    df_london = df_london.drop(columns=['postcode_area'])
    
    return df_london

## Load and Filter Loop

In [ ]:
# Process each yearly file
yearly_dfs = []

for csv_file in sorted(RAW_DIR.glob('pp-*.csv')):
    year = csv_file.stem.split('-')[1]
    print(f"\n Processing {csv_file.name}...")
    
    # Load in chunks to handle large files
    chunks = []
    chunk_size = 200_000
    
    for chunk in pd.read_csv(
        csv_file,
        header=None,
        names=COLUMNS,
        chunksize=chunk_size,
        low_memory=False,
        dtype={'price': 'int64'}
    ):
        # Filter to London immediately to save memory
        london_chunk = filter_london(chunk)
        chunks.append(london_chunk)
    
    # Combine chunks for this year
    year_df = pd.concat(chunks, ignore_index=True)
    year_df['source_year'] = int(year)
    
    print(f"    Loaded {len(year_df):,} London transactions for {year}")
    yearly_dfs.append(year_df)

# Merge all years
df_all = pd.concat(yearly_dfs, ignore_index=True)
print(f"\n{'='*50}")
print(f" TOTAL London transactions: {len(df_all):,}")
print(f" Date range: {df_all['date'].min()} to {df_all['date'].max()}")
print(f" Price range: £{df_all['price'].min():,} to £{df_all['price'].max():,}")
print(f"{'='*50}")


📂 Processing pp-2021.csv...
   ✅ Loaded 85,051 London transactions for 2021

📂 Processing pp-2022.csv...
   ✅ Loaded 75,217 London transactions for 2022

📂 Processing pp-2023.csv...
   ✅ Loaded 59,559 London transactions for 2023

📂 Processing pp-2024.csv...
   ✅ Loaded 65,964 London transactions for 2024

✅ TOTAL London transactions: 285,791
📅 Date range: 2021-01-01 00:00 to 2024-12-31 00:00
💷 Price range: £1 to £429,000,000


## Sanity Checks

In [5]:
# Shape and memory
print(f"Shape: {df_all.shape}")
print(f"Memory usage: {df_all.memory_usage(deep=True).sum() / 1024**2:.1f} MB\n")

# Records per year
print("Transactions per year:")
print(df_all['source_year'].value_counts().sort_index())

# First few rows
df_all.head()

Shape: (285791, 17)
Memory usage: 70.5 MB

Transactions per year:
source_year
2021    85051
2022    75217
2023    59559
2024    65964
Name: count, dtype: int64


,transaction_id,price,date,postcode,property_type,new_build,tenure,paon,saon,street,locality,town,district,county,ppd_type,record_status,source_year
0,{D93B27B0-3C8D-3100-E053-6C04A8C08887},467000,2021-02-15 00:00,NW2 3QB,F,N,L,40,FLAT 4,SHOOT UP HILL,NaN,LONDON,CAMDEN,GREATER LONDON,A,A,2021
1,{D93B27B0-3C8E-3100-E053-6C04A8C08887},2600000,2021-02-18 00:00,W1K 2SZ,F,N,L,94A,FLAT B,MOUNT STREET,NaN,LONDON,CITY OF WESTMINSTER,GREATER LONDON,A,A,2021
2,{D93B27B0-3C8F-3100-E053-6C04A8C08887},3070000,2021-02-11 00:00,NW3 7QN,F,N,L,36A,NaN,HOLLYCROFT AVENUE,NaN,LONDON,CAMDEN,GREATER LONDON,A,A,2021
3,{D93B27B0-3C90-3100-E053-6C04A8C08887},2176000,2021-03-09 00:00,W2 2HD,F,N,L,10,APARTMENT 2,FREDERICK CLOSE,NaN,LONDON,CITY OF WESTMINSTER,GREATER LONDON,A,A,2021
4,{D93B27B0-3C91-3100-E053-6C04A8C08887},2500000,2021-03-12 00:00,W2 2HD,F,N,L,10,APARTMENT 5,FREDERICK CLOSE,NaN,LONDON,CITY OF WESTMINSTER,GREATER LONDON,A,A,2021


## Save as Parquet

In [ ]:
output_path = PROCESSED_DIR / 'london_transactions_raw.parquet'
df_all.to_parquet(output_path, engine='pyarrow', compression='snappy', index=False)

# Check the saved file
size_mb = output_path.stat().st_size / (1024 * 1024)
print(f" Saved: {output_path}")
print(f" File size: {size_mb:.1f} MB")
print(f" Records: {len(df_all):,}")

✅ Saved: c:\Users\irfan\london-housing-analytics\data\processed\london_transactions_raw.parquet
📦 File size: 5.7 MB
📊 Records: 285,791


## sanity check (high level idea of data distribuition)

In [7]:
print("Transactions per year:")
print(df_all['source_year'].value_counts().sort_index())

print("\nProperty type distribution:")
print(df_all['property_type'].value_counts())

print("\nSample postcode areas (should all be London):")
print(df_all['postcode'].str.extract(r'^([A-Z]+)', expand=False).value_counts().head(10))

Transactions per year:
source_year
2021    85051
2022    75217
2023    59559
2024    65964
Name: count, dtype: int64

Property type distribution:
property_type
F    180511
T     65170
S     18001
O     17007
D      5102
Name: count, dtype: int64

Sample postcode areas (should all be London):
postcode
SW    65515
SE    57889
E     52914
N     42317
W     33447
NW    28069
EC     3734
WC     1906
Name: count, dtype: int64
